In [1]:
import pandas as pd
import numpy as np

from pathlib import Path

import warnings

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 180)

In [2]:
# ============================================================
# CONFIGURATION
# ============================================================

from pathlib import Path
import pandas as pd

# Project root
PROJECT_DIR = Path(
    r"D:\PycharmProjects\mf-ml"
)

# Data files
DATA_FILE = (
    PROJECT_DIR
    / "feature_selection_output"
    / "ml_feature_dataset.csv"
)

IC_FILE = (
    PROJECT_DIR
    / "feature_selection_output"
    / "ic_selected_features.csv"
)

# Output directory
OUTPUT_DIR = (
    PROJECT_DIR
    / "feature_selection_output"
)

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

# Research period
MODEL_START_MONTH = pd.Timestamp("2019-01-01")
MODEL_END_MONTH = pd.Timestamp("2025-12-01")

# Rolling framework
INITIAL_TRAIN_MONTHS = 60
OOS_MONTHS = 3
ROLL_MONTHS = 3

# Cross-validation
N_CV_SPLITS = 5

# Target
TARGET = "outperforms_benchmark"


# ------------------------------------------------------------
# VERIFY FILES
# ------------------------------------------------------------

print("DATA_FILE:")
print(DATA_FILE)
print("Exists:", DATA_FILE.exists())

print()

print("IC_FILE:")
print(IC_FILE)
print("Exists:", IC_FILE.exists())

assert DATA_FILE.exists(), (
    f"ML dataset not found:\n{DATA_FILE}"
)

assert IC_FILE.exists(), (
    f"IC feature file not found:\n{IC_FILE}"
)

print()
print("✓ Configuration and file paths are valid.")

DATA_FILE:
D:\PycharmProjects\mf-ml\feature_selection_output\ml_feature_dataset.csv
Exists: True

IC_FILE:
D:\PycharmProjects\mf-ml\feature_selection_output\ic_selected_features.csv
Exists: True

✓ Configuration and file paths are valid.


In [3]:
# ============================================================
# LOAD MODEL DATA
# ============================================================

model_df = pd.read_csv(DATA_FILE)

model_df["month_date"] = pd.to_datetime(
    model_df["month_date"]
)

model_df["target_month"] = pd.to_datetime(
    model_df["target_month"]
)

model_df = (
    model_df
    .sort_values(["month_date", "scheme_name"])
    .reset_index(drop=True)
)

print("model_df shape:", model_df.shape)
print(
    "Date range:",
    model_df["month_date"].min(),
    "→",
    model_df["month_date"].max()
)

display(model_df.head())

model_df shape: (7034, 29)
Date range: 2019-01-01 00:00:00 → 2026-06-01 00:00:00


,scheme_name,month_date,target_month,outperforms_benchmark,t_hml_250,t_hml_750,t_hml_180,sr_p1y,ir_p1y,ir_p6m,monthly_return_percent,t_const_180,sr_p6m,t_const_250,ir_p3y,const_90,const_180,t_const_90,t_const_120,const_250,t_hml_120,const_120,t_mkt_750,t_umd_750,t_hml_90,sr_p3y,r2_750,net_flow_percent,t_smb_250
0,Axis Flexi Cap Fund,2019-01-01,2019-02-01,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,DSP ELSS Tax Saver Fund,2019-01-01,2019-02-01,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,DSP Flexi Cap Fund,2019-01-01,2019-02-01,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,DSP Large & Mid Cap Fund,2019-01-01,2019-02-01,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Edelweiss ELSS Tax saver Fund,2019-01-01,2019-02-01,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [4]:
# ============================================================
# LOAD IC-SELECTED FEATURES
# ============================================================

ic_selected = pd.read_csv(
    IC_FILE
)

selected_features = (
    ic_selected[
        ic_selected["ic_pass"] == "PASS"
    ]["predictor"]
    .dropna()
    .tolist()
)

print(
    "Number of IC-selected features:",
    len(selected_features)
)

for i, feature in enumerate(
    selected_features,
    start=1
):
    print(
        f"{i:2d}. {feature}"
    )

Number of IC-selected features: 25
 1. t_hml_250
 2. t_hml_750
 3. t_hml_180
 4. sr_p1y
 5. ir_p1y
 6. ir_p6m
 7. monthly_return_percent
 8. t_const_180
 9. sr_p6m
10. t_const_250
11. ir_p3y
12. const_90
13. const_180
14. t_const_90
15. t_const_120
16. const_250
17. t_hml_120
18. const_120
19. t_mkt_750
20. t_umd_750
21. t_hml_90
22. sr_p3y
23. r2_750
24. net_flow_percent
25. t_smb_250


In [5]:
# ============================================================
# VALIDATE TARGET AND FEATURES
# ============================================================

assert TARGET in model_df.columns, (
    f"Target column '{TARGET}' not found."
)

missing_features = [
    f for f in selected_features
    if f not in model_df.columns
]

assert not missing_features, (
    f"Missing selected features: {missing_features}"
)

print("Target:", TARGET)
print()
print("Target distribution:")

display(
    model_df[TARGET]
    .value_counts(dropna=False)
    .to_frame("count")
)

print("\nTarget proportion:")

display(
    model_df[TARGET]
    .value_counts(
        normalize=True,
        dropna=False
    )
    .to_frame("proportion")
)

print("\n✓ All selected features exist.")

Target: outperforms_benchmark

Target distribution:


,count
outperforms_benchmark,
1,3614
0,3420



Target proportion:


,proportion
outperforms_benchmark,
1,0.51379
0,0.48621



✓ All selected features exist.


In [6]:
# ============================================================
# VERIFY ONE-MONTH-AHEAD TARGET
# ============================================================

target_check = (
    model_df[
        ["month_date", "target_month"]
    ]
    .drop_duplicates()
    .copy()
)

target_check["month_diff"] = (
    target_check["target_month"].dt.year * 12
    + target_check["target_month"].dt.month
    -
    (
        target_check["month_date"].dt.year * 12
        + target_check["month_date"].dt.month
    )
)

print("Month differences:")

display(
    target_check["month_diff"]
    .value_counts()
    .sort_index()
)

assert (
    target_check["month_diff"] == 1
).all(), (
    "Target month is not consistently one month ahead."
)

print("✓ Target is consistently one month ahead.")

Month differences:


month_diff
1    90
Name: count, dtype: int64

✓ Target is consistently one month ahead.


In [7]:
# ============================================================
# CREATE 8 ROLLING TRAIN / OOS WINDOWS
# ============================================================

available_months = pd.date_range(
    start=MODEL_START_MONTH,
    end=MODEL_END_MONTH,
    freq="MS"
)

rolling_windows = []

window_no = 1
train_start_idx = 0

while True:

    train_end_idx = (
        train_start_idx
        + INITIAL_TRAIN_MONTHS
        - 1
    )

    oos_start_idx = train_end_idx + 1

    oos_end_idx = (
        oos_start_idx
        + OOS_MONTHS
        - 1
    )

    if oos_end_idx >= len(available_months):
        break

    rolling_windows.append({
        "window": window_no,
        "train_start": available_months[train_start_idx],
        "train_end": available_months[train_end_idx],
        "oos_start": available_months[oos_start_idx],
        "oos_end": available_months[oos_end_idx]
    })

    window_no += 1
    train_start_idx += ROLL_MONTHS


rolling_windows_df = pd.DataFrame(
    rolling_windows
)

print(
    "Number of rolling windows:",
    len(rolling_windows_df)
)

display(rolling_windows_df)

Number of rolling windows: 8


,window,train_start,train_end,oos_start,oos_end
0,1,2019-01-01,2023-12-01,2024-01-01,2024-03-01
1,2,2019-04-01,2024-03-01,2024-04-01,2024-06-01
2,3,2019-07-01,2024-06-01,2024-07-01,2024-09-01
3,4,2019-10-01,2024-09-01,2024-10-01,2024-12-01
4,5,2020-01-01,2024-12-01,2025-01-01,2025-03-01
5,6,2020-04-01,2025-03-01,2025-04-01,2025-06-01
6,7,2020-07-01,2025-06-01,2025-07-01,2025-09-01
7,8,2020-10-01,2025-09-01,2025-10-01,2025-12-01


In [8]:
# ============================================================
# VALIDATE ROLLING WINDOWS
# ============================================================

for _, w in rolling_windows_df.iterrows():

    train = model_df[
        (model_df["month_date"] >= w["train_start"]) &
        (model_df["month_date"] <= w["train_end"])
    ]

    oos = model_df[
        (model_df["month_date"] >= w["oos_start"]) &
        (model_df["month_date"] <= w["oos_end"])
    ]

    train_usable = train.dropna(
        subset=selected_features + [TARGET]
    )

    oos_usable = oos.dropna(
        subset=selected_features
    )

    print(
        f"Window {int(w['window'])}: "
        f"TRAIN {w['train_start'].strftime('%Y-%m')} → "
        f"{w['train_end'].strftime('%Y-%m')} "
        f"({len(train_usable):,} usable) | "
        f"OOS {w['oos_start'].strftime('%Y-%m')} → "
        f"{w['oos_end'].strftime('%Y-%m')} "
        f"({len(oos_usable):,} usable)"
    )

Window 1: TRAIN 2019-01 → 2023-12 (646 usable) | OOS 2024-01 → 2024-03 (96 usable)
Window 2: TRAIN 2019-04 → 2024-03 (742 usable) | OOS 2024-04 → 2024-06 (97 usable)
Window 3: TRAIN 2019-07 → 2024-06 (839 usable) | OOS 2024-07 → 2024-09 (99 usable)
Window 4: TRAIN 2019-10 → 2024-09 (938 usable) | OOS 2024-10 → 2024-12 (126 usable)
Window 5: TRAIN 2020-01 → 2024-12 (1,064 usable) | OOS 2025-01 → 2025-03 (220 usable)
Window 6: TRAIN 2020-04 → 2025-03 (1,284 usable) | OOS 2025-04 → 2025-06 (226 usable)
Window 7: TRAIN 2020-07 → 2025-06 (1,510 usable) | OOS 2025-07 → 2025-09 (235 usable)
Window 8: TRAIN 2020-10 → 2025-09 (1,745 usable) | OOS 2025-10 → 2025-12 (247 usable)


In [9]:
# ============================================================
# FINAL PREPARATION CHECK
# ============================================================

assert len(rolling_windows_df) == 8, (
    "Expected exactly 8 rolling windows."
)

assert (
    rolling_windows_df["train_end"]
    < rolling_windows_df["oos_start"]
).all()

assert (
    rolling_windows_df["oos_end"]
    <= MODEL_END_MONTH
).all()

print("=" * 70)
print("10A PREPARATION COMPLETE")
print("=" * 70)

print(f"Model data:       {model_df.shape}")
print(f"Selected features: {len(selected_features)}")
print(f"Target:             {TARGET}")
print(f"Rolling windows:    {len(rolling_windows_df)}")
print(f"CV folds:            {N_CV_SPLITS}")

print("\nObjects ready for model notebooks:")
print("  model_df")
print("  selected_features")
print("  TARGET")
print("  rolling_windows_df")

10A PREPARATION COMPLETE
Model data:       (7034, 29)
Selected features: 25
Target:             outperforms_benchmark
Rolling windows:    8
CV folds:            5

Objects ready for model notebooks:
  model_df
  selected_features
  TARGET
  rolling_windows_df
